# BPL OCR · pages 161–167

One pipeline, the same seven scans and shared detected line coordinates, five selectable models.
Run one model in each **fresh hosted Colab runtime** and upload the latest data ZIP to accumulate outputs.
No local OCR installation or API key is required. The scope is OCR text only.

The supplied folder already includes the existing Tesseract and dots.mocr outputs.
Other model columns remain empty until run. Historical Tesseract text used full-page context;
fresh recognition in this notebook uses the same line crops as the other models.
One detected line is not necessarily one physical printed line or one person.

## 1. Choose a model and set up Colab

Set `ENGINE` before running. Use GPU for dots.mocr and Qwen; EasyOCR uses an available GPU.
Tesseract and this PaddleOCR setup use CPU. Reset the runtime before changing the model.

In [ ]:
ENGINE = 'easyocr'  # tesseract, dots_mocr, easyocr, paddleocr, qwen3_vl_2b

import subprocess
import sys
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('Use a hosted Google Colab runtime.')
if ENGINE not in ('tesseract', 'dots_mocr', 'easyocr', 'paddleocr', 'qwen3_vl_2b'):
    raise ValueError('Choose one of the five documented model names.')
if '_SETUP_ENGINE' in globals() and ENGINE != _SETUP_ENGINE:
    raise RuntimeError('Reset the runtime before changing ENGINE.')
dependencies = ['Pillow', 'numpy']
if ENGINE == 'tesseract':
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'], check=True)
elif ENGINE == 'easyocr':
    dependencies += ['easyocr==1.7.2']
elif ENGINE == 'paddleocr':
    dependencies += ['paddlepaddle==3.2.0', 'paddleocr==3.7.0', 'huggingface-hub==0.36.2']
else:
    dependencies += ['transformers==4.57.6', 'accelerate==1.15.0', 'huggingface-hub==0.36.2']
    if ENGINE == 'dots_mocr':
        dependencies += ['qwen-vl-utils==0.0.14']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *dependencies], check=True)
DEVICE = 'cpu' if ENGINE in ('tesseract', 'paddleocr') else 'auto'
if ENGINE in ('dots_mocr', 'qwen3_vl_2b'):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select Runtime > Change runtime type > GPU.')
_SETUP_ENGINE = ENGINE
print('Selected:', ENGINE, '| device:', DEVICE)

In [ ]:
# Exact shared ocr_pipeline.py; no repository checkout is needed.
_PIPELINE_SOURCE = '"""One BPL OCR pipeline: common scans/line geometry, five engines, line text only.\n\nHosted Colab runs recognition. Local CLI only builds the notebook or packages\nthe common data folder. No accuracy scoring, field extraction or gold labels.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nimport zipfile\n\nFORMAT = "bpl-ocr-lines-v1"\nENGINES = ("tesseract", "dots_mocr", "easyocr", "paddleocr", "qwen3_vl_2b")\nTEXT_FIELDS = ("tesseract_text", "mocr_text", "easyocr_text", "paddleocr_text", "qwen_text")\nPAGES = list(range(161, 168))\n\n\ndef sha_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef sha_text(text):\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef identity(value):\n    return sha_text(json.dumps(value, sort_keys=True, ensure_ascii=False))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef source_identity(index):\n    return identity({"pages": index["pages"], "lines": index["lines"], "scope": index["scope"]})\n\n\ndef load_sources(root):\n    root = Path(root)\n    index = json.loads((root / "source/index.json").read_text(encoding="utf-8"))\n    if index.get("format") != FORMAT or [p["printed_page"] for p in index["pages"]] != PAGES:\n        raise ValueError("Expected the common printed-page 161–167 source folder")\n    if index.get("source_id") != source_identity(index):\n        raise ValueError("Source metadata or shared line geometry changed")\n    from PIL import Image\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    for p in index["pages"]:\n        if p["image"] != f"source/p{p[\'printed_page\']}.jpg":\n            raise ValueError("Unexpected source image path")\n        image_path = root / p["image"]\n        if sha_file(image_path) != p["sha256"]:\n            raise ValueError("Source image changed: " + p["image"])\n        with Image.open(image_path) as image:\n            if list(image.size) != [p["width"], p["height"]]:\n                raise ValueError("Source dimensions changed")\n            image.verify()\n    seen = set()\n    for row in index["lines"]:\n        page = pages[row["printed_page"]]\n        x0, y0, x1, y1 = row["bbox"]\n        if (row["line_id"] in seen or not (0 <= x0 < x1 <= page["width"] and 0 <= y0 < y1 <= page["height"])):\n            raise ValueError("Duplicate line ID or invalid source coordinates")\n        seen.add(row["line_id"])\n    return index\n\n\ndef visible_lines(index):\n    """Keep existing order; page 161 excludes the abbreviation key and footer."""\n    return [r for r in index["lines"] if r["printed_page"] != 161 or 1645 <= r["bbox"][1] < 2678]\n\n\ndef line_identity(index, row):\n    page = next(p for p in index["pages"] if p["printed_page"] == row["printed_page"])\n    return identity({"source_sha256": page["sha256"], "line_id": row["line_id"], "bbox": row["bbox"]})\n\n\ndef read_jsonl(path):\n    result = []\n    if Path(path).exists():\n        for line in Path(path).read_text(encoding="utf-8").splitlines():\n            try:\n                value = json.loads(line)\n                if isinstance(value, dict):\n                    result.append(value)\n            except ValueError:\n                pass  # Retain usable records when an interrupted append is incomplete.\n    return result\n\n\ndef read_predictions(root, index, engine):\n    if engine not in ENGINES:\n        raise ValueError("Unknown engine")\n    expected = {r["line_id"]: r for r in index["lines"]}\n    models_path = Path(root) / "results/models.json"\n    models = json.loads(models_path.read_text()) if models_path.exists() else {}\n    metadata = models.get(engine, {})\n    result = {}\n    for record in read_jsonl(Path(root) / "results/raw" / (engine + ".jsonl")):\n        row = expected.get(record.get("line_id"))\n        text = record.get("raw_text")\n        if row is None or not isinstance(text, str) or sha_text(text) != record.get("raw_text_sha256"):\n            continue\n        if record.get("input_id") is not None:\n            matches = record["input_id"] == line_identity(index, row)\n        else:\n            # Native v4 mOCR log is preserved byte-for-byte during consolidation.\n            matches = (metadata.get("imported_source_id") == index["source_id"]\n                       and record.get("bbox") == row["bbox"])\n        if matches and record.get("status") in ("completed", "empty", "failed", "truncated", "out_of_memory", "detected_no_text"):\n            result[row["line_id"]] = record\n    return result\n\n\ndef export_lines(root, index=None):\n    root = Path(root)\n    index = load_sources(root) if index is None else index\n    predictions = {e: read_predictions(root, index, e) for e in ENGINES}\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    path = root / "results/lines.csv"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    counts = {}\n    with path.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[*TEXT_FIELDS, "sources"])\n        writer.writeheader()\n        for row in visible_lines(index):\n            out = {field: " ".join(predictions[e].get(row["line_id"], {}).get("raw_text", "").split())\n                   for e, field in zip(ENGINES, TEXT_FIELDS)}\n            out["sources"] = pages[row["printed_page"]]["source_page_url"]\n            writer.writerow(out)\n    for engine in ENGINES:\n        counts[engine] = {status: sum(predictions[engine].get(r["line_id"], {}).get("status", "not_run") == status\n                                     for r in visible_lines(index))\n                          for status in ("completed", "empty", "failed", "truncated", "out_of_memory", "detected_no_text", "not_run")}\n    write_json(root / "results/status.json", {"format": FORMAT, "source_id": index["source_id"],\n               "rows": len(visible_lines(index)), "models": counts,\n               "meaning": "One shared detected scan line per row, not one person. Blank text: inspect raw/status records. No accuracy measures."})\n    return path\n\n\ndef import_bundle(archive, root):\n    root = Path(root)\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError("Output folder is populated. Continue with the loaded ROOT, or use a fresh Colab runtime.")\n    with tempfile.TemporaryDirectory() as temporary:\n        staging = Path(temporary) / "data"\n        with zipfile.ZipFile(archive) as z:\n            files = [i for i in z.infolist() if not i.is_dir() and not any(p.startswith(".") or p == "__MACOSX" for p in Path(i.filename).parts)]\n            indexes = [i.filename for i in files if i.filename.endswith("source/index.json")]\n            if len(indexes) != 1 or sum(i.file_size for i in files) > 500_000_000:\n                raise ValueError("Upload one common BPL OCR folder ZIP (source/index.json required)")\n            prefix = indexes[0][:-len("source/index.json")]\n            for info in files:\n                if not info.filename.startswith(prefix):\n                    continue\n                relative = Path(info.filename[len(prefix):])\n                if relative.is_absolute() or ".." in relative.parts:\n                    raise ValueError("Unsafe ZIP path")\n                if (not relative.parts or relative.parts[0] not in ("source", "results")\n                        or relative.suffix not in (".jpg", ".json", ".jsonl", ".csv", ".tsv", ".txt")):\n                    continue\n                if info.file_size > 100_000_000:\n                    raise ValueError("ZIP member too large")\n                target = staging / relative\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(z.read(info))\n        index = load_sources(staging)\n        shutil.copytree(staging, root, dirs_exist_ok=True)\n    export_lines(root, index)\n    return index\n\n\ndef pack_bundle(root, destination):\n    """Create an on-demand transfer ZIP; keep source images once in the workspace."""\n    root, destination = Path(root), Path(destination)\n    if root.resolve() in destination.resolve().parents:\n        raise ValueError("Write the transfer ZIP outside the data folder")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".part")\n    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:\n        for directory in ("source", "results"):\n            for path in sorted((root / directory).rglob("*")):\n                if path.is_file() and not any(p.startswith(".") for p in path.relative_to(root).parts):\n                    archive.write(path, path.relative_to(root))\n    temporary.replace(destination)\n    return destination\n\n\ndef run_ocr(root, engine, *, device="auto", max_new_lines=None, force=False, adapter_factory=None):\n    """Append exact per-line outputs and checkpoint the same CSV for every model."""\n    root = Path(root)\n    if engine not in ENGINES:\n        raise ValueError("Choose one of: " + ", ".join(ENGINES))\n    if max_new_lines is not None and (type(max_new_lines) is not int or max_new_lines <= 0):\n        raise ValueError("max_new_lines must be None or a positive integer")\n    index = load_sources(root)\n    lines = visible_lines(index)\n    old = read_predictions(root, index, engine)\n    pending = [r for r in lines if force or old.get(r["line_id"], {}).get("status") != "completed"]\n    if max_new_lines is not None:\n        pending = pending[:max_new_lines]\n    print(engine, ":", len(lines), "source lines;", len(pending), "to recognize this run.", flush=True)\n    if not pending:\n        export_lines(root, index)\n        return {"engine": engine, "new_lines": 0, "message": "Existing completed outputs retained; no model loaded."}\n    models_path = root / "results/models.json"\n    models = json.loads(models_path.read_text()) if models_path.exists() else {}\n    prior = models.get(engine, {})\n    pinned = prior.get("settings_history", {}).get(prior.get("active_settings_id"), {})\n    adapter = (adapter_factory or build_recognizer)(engine, device=device, pinned_settings=pinned,\n                                                   model_cache=root.parent / "bpl_ocr_model_cache")\n    settings = {**adapter.settings, "pipeline_code_sha256": sha_file(__file__)}\n    settings_id = identity(settings)\n    history = dict(prior.get("settings_history", {}))\n    history[settings_id] = settings\n    models[engine] = {**prior, "active_settings_id": settings_id, "settings_history": history,\n                      "last_runtime": adapter.metadata,\n                      "input": "Shared original scan line crops. Existing imports retain their historical context in settings_history."}\n    write_json(models_path, models)\n    pages = {p["printed_page"]: p for p in index["pages"]}\n    log_path = root / "results/raw" / (engine + ".jsonl")\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    active_page, image, attempted = None, None, 0\n    from PIL import Image\n    try:\n        with log_path.open("a", encoding="utf-8") as handle:\n            if log_path.stat().st_size and not log_path.read_bytes().endswith(b"\\n"):\n                handle.write("\\n")\n            for row in pending:\n                if active_page != row["printed_page"]:\n                    if image is not None:\n                        image.close()\n                    page = pages[row["printed_page"]]\n                    if sha_file(root / page["image"]) != page["sha256"]:\n                        raise ValueError("Source image changed during OCR")\n                    with Image.open(root / page["image"]) as source:\n                        image = source.convert("RGB")\n                    active_page = row["printed_page"]\n                record = {"line_id": row["line_id"], "bbox": row["bbox"], "input_id": line_identity(index, row),\n                          "settings_id": settings_id}\n                try:\n                    with image.crop(tuple(row["bbox"])) as tile:\n                        result = adapter.predict(tile)\n                    if not isinstance(result.get("raw_text"), str) or result.get("status") not in ("completed", "empty", "truncated"):\n                        raise ValueError("Unexpected recognition response")\n                    record.update(result)\n                except Exception as error:\n                    oom = "out of memory" in str(error).lower() or "outofmemory" in type(error).__name__.lower()\n                    record.update(raw_text="", status="out_of_memory" if oom else "failed",\n                                  error=type(error).__name__ + ": " + str(error))\n                record["raw_text_sha256"] = sha_text(record["raw_text"])\n                handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                handle.flush()\n                attempted += 1\n                print(engine, row["line_id"], record["status"], f"({attempted}/{len(pending)})", flush=True)\n                if attempted % 20 == 0:\n                    export_lines(root, index)\n                if record["status"] == "out_of_memory":\n                    break\n    except KeyboardInterrupt:\n        print("Interrupted. Completed lines are saved. Run the download cell before leaving Colab.")\n    finally:\n        if image is not None:\n            image.close()\n        try:\n            adapter.close()\n        finally:\n            export_lines(root, index)\n    return {"engine": engine, "new_lines": attempted, "csv": str(root / "results/lines.csv")}\n\n\n"""Recognition adapters for the common BPL line-image pipeline.\n\nOnly standard-library imports execute at import time. OCR frameworks and model\nweights are loaded only by build_recognizer(), in the hosted Colab runtime.\nEvery recognizer sees the same original source crop, never another OCR\'s text.\n"""\nimport base64\nimport gc\nimport hashlib\nimport importlib.metadata\nimport io\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport urllib.request\n\nRECOGNIZER_KEYS = ("tesseract", "dots_mocr", "easyocr", "paddleocr", "qwen3_vl_2b")\nTESSDATA_REVISION = "e12c65a915945e4c28e237a9b52bc4a8f39a0cec"\nTESSDATA_SHA256 = "8280aed0782fe27257a68ea10fe7ef324ca0f8d85bd2fd145d1c2b560bcb66ba"\nDOTS_REVISION = "e539fbb52280393adc081b289ec597430a0f9031"\nDOTS_PROMPT = "Extract the text content from this image."\n\n\ndef _model_packages(*names):\n    result = {}\n    for name in names:\n        try:\n            result[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            result[name] = None\n    return result\n\n\ndef _model_file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _model_revision(hub, repo, stored=None):\n    revision = stored if stored is not None else hub.model_info(repo).sha\n    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):\n        raise ValueError("Missing or invalid immutable model revision: " + repo)\n    return revision\n\n\ndef _recognized(raw_text, **extra):\n    return {"raw_text": raw_text, "status": "completed" if raw_text.strip() else "empty", **extra}\n\n\ndef _generation_result(raw, tokens, eos, limit):\n    eos = eos if isinstance(eos, list) else [eos]\n    truncated = len(tokens) >= limit and (not tokens or tokens[-1] not in eos)\n    return {"raw_text": raw,\n            "status": "truncated" if truncated else ("completed" if raw.strip() else "empty"),\n            "extra": {"generated_tokens": len(tokens), "generated_token_ids": tokens}}\n\n\nclass _LineRecognizer:\n    def __init__(self, engine, device, framework=None, packages=None):\n        self.engine, self.device, self.framework = engine, device, framework\n        self.gpu = device.startswith(("gpu", "cuda"))\n        self.settings = {"engine": engine, "adapter_version": 1, "input": "one original RGB source-line crop",\n                         "device_type": "cuda" if self.gpu else "cpu", "precision": "float32",\n                         "packages": packages or {}}\n        self.metadata = {"actual_device": device, "platform": platform.platform(),\n                         "processor": platform.processor(), "packages": packages or {}}\n        self.model = None\n\n    def close(self):\n        if self.engine == "paddleocr" and hasattr(self.model, "close"):\n            self.model.close()\n        self.model = None\n        if hasattr(self, "processor"):\n            self.processor = None\n        gc.collect()\n        if self.gpu:\n            cuda = self.framework.device.cuda if self.engine == "paddleocr" else self.framework.cuda\n            if hasattr(cuda, "empty_cache"):\n                cuda.empty_cache()\n\n\nclass _TesseractRecognizer(_LineRecognizer):\n    def __init__(self, model_cache):\n        super().__init__("tesseract", "cpu", packages=_model_packages("Pillow"))\n        self.binary = shutil.which("tesseract")\n        if not self.binary:\n            raise RuntimeError("Run the Colab Tesseract setup cell first")\n        self.data_dir = Path(model_cache) / "tessdata_best" / TESSDATA_REVISION\n        self.data_dir.mkdir(parents=True, exist_ok=True)\n        target = self.data_dir / "eng.traineddata"\n        if not target.exists() or _model_file_sha(target) != TESSDATA_SHA256:\n            url = f"https://raw.githubusercontent.com/tesseract-ocr/tessdata_best/{TESSDATA_REVISION}/eng.traineddata"\n            request = urllib.request.Request(url, headers={"User-Agent": "BPL-OCR-Pipeline/1.0"})\n            partial = target.with_suffix(".part")\n            try:\n                with urllib.request.urlopen(request, timeout=60) as response, partial.open("wb") as handle:\n                    shutil.copyfileobj(response, handle)\n                if _model_file_sha(partial) != TESSDATA_SHA256:\n                    raise ValueError("Pinned English Tesseract weights checksum mismatch")\n                partial.replace(target)\n            finally:\n                partial.unlink(missing_ok=True)\n        version = subprocess.run([self.binary, "--version"], capture_output=True, text=True, check=True).stdout.splitlines()[0]\n        self.settings.update({"tesseract_version": version, "language": "eng", "oem": 1, "psm": 7,\n                              "english_revision": TESSDATA_REVISION, "english_sha256": TESSDATA_SHA256,\n                              "preprocessing": {"mode": "L", "scale": 2, "resampling": "LANCZOS", "white_border": 10}})\n\n    def predict(self, image):\n        from PIL import Image, ImageOps\n        crop = image.convert("L")\n        crop = crop.resize((crop.width*2, crop.height*2), Image.Resampling.LANCZOS)\n        crop = ImageOps.expand(crop, border=10, fill=255)\n        encoded = io.BytesIO()\n        crop.save(encoded, format="PNG")\n        result = subprocess.run([self.binary, "stdin", "stdout", "--tessdata-dir", str(self.data_dir),\n                                 "-l", "eng", "--oem", "1", "--psm", "7"],\n                                input=encoded.getvalue(), capture_output=True, check=True)\n        return _recognized(result.stdout.decode("utf-8"), extra={"stderr": result.stderr.decode("utf-8", errors="replace")})\n\n\nclass _EasyRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, np, easyocr, model_cache):\n        super().__init__("easyocr", device, torch, _model_packages("easyocr", "torch", "torchvision", "numpy", "Pillow"))\n        self.np = np\n        self.model = easyocr.Reader(["en"], gpu=device if self.gpu else False,\n                                    detector=False, recog_network="english_g2", quantize=False,\n                                    cudnn_benchmark=False, model_storage_directory=str(Path(model_cache) / "easyocr"))\n        model_file = Path(self.model.model_storage_directory) / "english_g2.pth"\n        self.settings.update({"recognizer": "english_g2", "languages": ["en"], "detector": False,\n            "quantize": False, "cudnn_benchmark": False, "model_sha256": _model_file_sha(model_file),\n            "recognition": {"decoder": "greedy", "batch_size": 1, "workers": 0, "detail": 1,\n                            "paragraph": False, "contrast_ths": 0.1, "adjust_contrast": 0.5, "filter_ths": 0.003}})\n        self.metadata["actual_device"] = str(self.model.device)\n        if self.gpu:\n            self.metadata["gpu_name"] = torch.cuda.get_device_name()\n\n    def predict(self, image):\n        predictions = self.model.recognize(self.np.asarray(image.convert("L")), **self.settings["recognition"])\n        records = [{"polygon": [[float(x), float(y)] for x, y in box],\n                    "text": str(text), "confidence": float(confidence)} for box, text, confidence in predictions]\n        return _recognized("\\n".join(r["text"] for r in records), extra={"recognition_results": records})\n\n\nclass _PaddleRecognizer(_LineRecognizer):\n    def __init__(self, device, paddle, np, paddleocr, hub, pinned_settings=None):\n        super().__init__("paddleocr", device, paddle,\n                         _model_packages("paddleocr", "paddlex", "paddlepaddle", "paddlepaddle-gpu", "numpy", "Pillow"))\n        self.np = np\n        paddle.set_device(device)\n        repo, name = "PaddlePaddle/en_PP-OCRv5_mobile_rec", "en_PP-OCRv5_mobile_rec"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision"))\n        folder = Path(hub.snapshot_download(repo, revision=revision,\n                      allow_patterns=["config.json", "inference.json", "inference.yml", "inference.pdiparams"]))\n        hashes = {p.name: _model_file_sha(p) for p in sorted(folder.iterdir()) if p.is_file()}\n        backend = {"enable_hpi": False, "engine": "paddle_static", "use_tensorrt": False,\n                   "precision": "fp32", "enable_mkldnn": False, "cpu_threads": 2, "enable_cinn": False}\n        self.model = paddleocr.TextRecognition(model_name=name, model_dir=str(folder), device=device, **backend)\n        self.settings.update({"model": repo, "recognizer": name, "revision": revision,\n                              "model_sha256": hashes, "input_channels": "BGR", "batch_size": 1, "backend": backend})\n        self.metadata["actual_device"] = str(paddle.get_device())\n        if self.gpu:\n            self.metadata["gpu_name"] = paddle.device.cuda.get_device_name()\n\n    def predict(self, image):\n        bgr = self.np.asarray(image.convert("RGB"))[:, :, ::-1].copy()\n        results = list(self.model.predict(input=bgr, batch_size=1))\n        if len(results) != 1:\n            raise ValueError("Expected one Paddle recognition result for one source crop")\n        return _recognized(str(results[0]["rec_text"]), extra={"confidence": float(results[0]["rec_score"])})\n\n\nclass _QwenRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, transformers, hub, pinned_settings=None):\n        super().__init__("qwen3_vl_2b", device, torch,\n                         _model_packages("transformers", "torch", "torchvision", "accelerate", "numpy", "Pillow"))\n        repo = "Qwen/Qwen3-VL-2B-Instruct"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision"))\n        dtype = torch.float16 if self.gpu else torch.float32\n        self.model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(\n            repo, revision=revision, dtype=dtype, attn_implementation="sdpa", device_map=device).eval()\n        self.processor = transformers.AutoProcessor.from_pretrained(repo, revision=revision,\n                             use_fast=False, min_pixels=4096, max_pixels=262144)\n        prompt = ("Transcribe all text in this image faithfully. Preserve printed ditto marks, "\n                  "abbreviations, punctuation and line breaks. Do not expand, correct, infer, "\n                  "or explain anything. Output only the transcription.")\n        self.settings.update({"model": repo, "revision": revision, "dtype": str(dtype),\n            "precision": "float16" if self.gpu else "float32", "attention": "sdpa", "prompt": prompt,\n            "min_pixels": 4096, "max_pixels": 262144,\n            "generation": {"max_new_tokens": 256, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.0},\n            "processor_sha256": hashlib.sha256(self.processor.image_processor.to_json_string().encode()).hexdigest(),\n            "generation_config": self.model.generation_config.to_dict()})\n        if self.gpu:\n            self.metadata["gpu_name"] = torch.cuda.get_device_name()\n\n    def predict(self, image):\n        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": self.settings["prompt"]}]}]\n        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = self.processor(text=[text], images=[image.convert("RGB")], return_tensors="pt").to(self.model.device)\n        inputs.pop("token_type_ids", None)\n        with self.framework.inference_mode():\n            generated = self.model.generate(**inputs, **self.settings["generation"])\n        tokens = generated[0, inputs["input_ids"].shape[-1]:].tolist()\n        raw = self.processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n        return _generation_result(raw, tokens, self.model.generation_config.eos_token_id, 256)\n\n\nclass _DotsRecognizer(_LineRecognizer):\n    def __init__(self, device, torch, transformers, hub, model_cache, pinned_settings=None):\n        if not device.startswith("cuda"):\n            raise RuntimeError("dots.mocr requires a Colab GPU runtime")\n        super().__init__("dots_mocr", device, torch,\n                         _model_packages("torch", "transformers", "qwen-vl-utils", "accelerate", "huggingface-hub", "Pillow"))\n        repo = "dots-studio/dots.mocr"\n        revision = _model_revision(hub, repo, (pinned_settings or {}).get("revision", DOTS_REVISION))\n        local = Path(model_cache) / "dots_mocr" / revision\n        hub.snapshot_download(repo, revision=revision, local_dir=str(local))\n        hub.hf_hub_download(repo, "modeling_dots_vision.py", revision=revision, local_dir=str(local), force_download=True)\n        vision_path = local / "modeling_dots_vision.py"\n        source = vision_path.read_text()\n        original_sha = hashlib.sha256(source.encode()).hexdigest()\n        if \'"sdpa"\' not in source and "\'sdpa\'" not in source:\n            raise RuntimeError("The pinned dots vision code lacks the expected SDPA implementation")\n        guarded = "try:\\n    from flash_attn import flash_attn_varlen_func\\nexcept ImportError:\\n    flash_attn_varlen_func = None\\n"\n        if "from flash_attn import flash_attn_varlen_func" in source.splitlines():\n            vision_path.write_text(source.replace("from flash_attn import flash_attn_varlen_func", guarded, 1))\n        config = transformers.AutoConfig.from_pretrained(str(local), trust_remote_code=True)\n        if isinstance(config.vision_config, dict):\n            config.vision_config["attn_implementation"] = "sdpa"\n        else:\n            config.vision_config.attn_implementation = "sdpa"\n        self.model = transformers.AutoModelForCausalLM.from_pretrained(str(local), config=config,\n            trust_remote_code=True, torch_dtype=torch.bfloat16, attn_implementation="sdpa",\n            device_map={"": device}, low_cpu_mem_usage=True).eval()\n        self.processor = transformers.AutoProcessor.from_pretrained(str(local), trust_remote_code=True)\n        self.settings.update({"model": repo, "revision": revision, "prompt": DOTS_PROMPT,\n            "dtype": "torch.bfloat16", "precision": "bfloat16", "attention": "sdpa", "max_pixels": 262144,\n            "generation": {"max_new_tokens": 256, "do_sample": False},\n            "original_vision_code_sha256": original_sha, "used_vision_code_sha256": _model_file_sha(vision_path)})\n        self.metadata["gpu_name"] = torch.cuda.get_device_name()\n        self.metadata["native_bfloat16_supported"] = bool(torch.cuda.is_bf16_supported())\n\n    def predict(self, image):\n        from qwen_vl_utils import process_vision_info\n        encoded = io.BytesIO()\n        image.save(encoded, format="PNG")\n        data_url = "data:image/png;base64," + base64.b64encode(encoded.getvalue()).decode()\n        messages = [{"role": "user", "content": [\n            {"type": "image", "image": data_url, "max_pixels": 262144},\n            {"type": "text", "text": DOTS_PROMPT}]}]\n        chat = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        images, videos = process_vision_info(messages)\n        inputs = self.processor(text=[chat], images=images, videos=videos, padding=True, return_tensors="pt").to(self.device)\n        with self.framework.inference_mode():\n            output = self.model.generate(**inputs, **self.settings["generation"])\n        tokens = output[0, inputs.input_ids.shape[1]:].tolist()\n        raw = self.processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n        result = _generation_result(raw, tokens, self.model.generation_config.eos_token_id, 256)\n        result["extra"]["processor_sizes"] = [list(im.size) for im in images]\n        return result\n\n\ndef build_recognizer(engine, device="auto", pinned_settings=None, model_cache=None):\n    """Load one recognition engine in hosted Colab; preserve immutable resume pins."""\n    if engine not in RECOGNIZER_KEYS:\n        raise ValueError("Unknown OCR engine: " + str(engine))\n    if device not in {"auto", "cpu", "gpu", "gpu:0", "cuda", "cuda:0"}:\n        raise ValueError("device must be auto, cpu or cuda:0")\n    model_cache = Path(model_cache or "/content/bpl_model_cache")\n    model_cache.mkdir(parents=True, exist_ok=True)\n    if engine == "tesseract":\n        return _TesseractRecognizer(model_cache)\n    if engine == "paddleocr":\n        import paddle\n        import paddleocr\n        import numpy as np\n        import huggingface_hub as hub\n        if device == "auto":\n            device = "gpu:0" if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() else "cpu"\n        return _PaddleRecognizer("cpu" if device == "cpu" else "gpu:0", paddle, np, paddleocr, hub, pinned_settings)\n    import torch\n    if device == "auto":\n        device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    device = "cpu" if device == "cpu" else "cuda:0"\n    if device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU was requested but is not available")\n    if engine == "easyocr":\n        import easyocr\n        import numpy as np\n        return _EasyRecognizer(device, torch, np, easyocr, model_cache)\n    import transformers\n    import huggingface_hub as hub\n    if engine == "dots_mocr":\n        return _DotsRecognizer(device, torch, transformers, hub, model_cache, pinned_settings)\n    return _QwenRecognizer(device, torch, transformers, hub, pinned_settings)\n\n\n\ndef build_notebook():\n    """Embed this exact shared file into the sole hosted Colab OCR notebook."""\n    import nbformat as nbf\n    source = Path(__file__).read_text(encoding="utf-8")\n    cells = []\n    def md(text):\n        cells.append(nbf.v4.new_markdown_cell(text.strip()))\n    def code(text, name):\n        cell = nbf.v4.new_code_cell(text.strip())\n        cell["id"] = "ocr-" + name\n        cells.append(cell)\n    md("""# BPL OCR · pages 161–167\n\nOne pipeline, the same seven scans and shared detected line coordinates, five selectable models.\nRun one model in each **fresh hosted Colab runtime** and upload the latest data ZIP to accumulate outputs.\nNo local OCR installation or API key is required. The scope is OCR text only.\n\nThe supplied folder already includes the existing Tesseract and dots.mocr outputs.\nOther model columns remain empty until run. Historical Tesseract text used full-page context;\nfresh recognition in this notebook uses the same line crops as the other models.\nOne detected line is not necessarily one physical printed line or one person.\n""")\n    md("""## 1. Choose a model and set up Colab\n\nSet `ENGINE` before running. Use GPU for dots.mocr and Qwen; EasyOCR uses an available GPU.\nTesseract and this PaddleOCR setup use CPU. Reset the runtime before changing the model.\n""")\n    code("""ENGINE = \'easyocr\'  # tesseract, dots_mocr, easyocr, paddleocr, qwen3_vl_2b\n\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nif not Path(\'/content\').exists():\n    raise RuntimeError(\'Use a hosted Google Colab runtime.\')\nif ENGINE not in (\'tesseract\', \'dots_mocr\', \'easyocr\', \'paddleocr\', \'qwen3_vl_2b\'):\n    raise ValueError(\'Choose one of the five documented model names.\')\nif \'_SETUP_ENGINE\' in globals() and ENGINE != _SETUP_ENGINE:\n    raise RuntimeError(\'Reset the runtime before changing ENGINE.\')\ndependencies = [\'Pillow\', \'numpy\']\nif ENGINE == \'tesseract\':\n    subprocess.run([\'apt-get\', \'update\', \'-qq\'], check=True)\n    subprocess.run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'tesseract-ocr\'], check=True)\nelif ENGINE == \'easyocr\':\n    dependencies += [\'easyocr==1.7.2\']\nelif ENGINE == \'paddleocr\':\n    dependencies += [\'paddlepaddle==3.2.0\', \'paddleocr==3.7.0\', \'huggingface-hub==0.36.2\']\nelse:\n    dependencies += [\'transformers==4.57.6\', \'accelerate==1.15.0\', \'huggingface-hub==0.36.2\']\n    if ENGINE == \'dots_mocr\':\n        dependencies += [\'qwen-vl-utils==0.0.14\']\nsubprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--disable-pip-version-check\', *dependencies], check=True)\nDEVICE = \'cpu\' if ENGINE in (\'tesseract\', \'paddleocr\') else \'auto\'\nif ENGINE in (\'dots_mocr\', \'qwen3_vl_2b\'):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select Runtime > Change runtime type > GPU.\')\n_SETUP_ENGINE = ENGINE\nprint(\'Selected:\', ENGINE, \'| device:\', DEVICE)\n""", "setup")\n    code("# Exact shared ocr_pipeline.py; no repository checkout is needed.\\n"\n         + "_PIPELINE_SOURCE = " + repr(source) + "\\n"\n         + "_CODE = Path(\'/content/bpl_ocr_code\')\\n_CODE.mkdir(exist_ok=True)\\n"\n         + "(_CODE / \'ocr_pipeline.py\').write_text(_PIPELINE_SOURCE, encoding=\'utf-8\')\\n"\n         + "sys.path.insert(0, str(_CODE))\\n"\n         + "from ocr_pipeline import import_bundle, run_ocr, export_lines, pack_bundle\\n", "helpers")\n    md("""## 2. Upload the common source/results ZIP\n\nUpload a ZIP of the common `fa26/data/ocr/` folder, or the latest ZIP downloaded below.\nIt contains `source/` and `results/`. Seven scans are kept once; line crops exist in memory only.\nThe existing coordinates are shared by all models, including any detection mistakes.\nPage 161 uses its lower directory body; pages 162–167 retain their existing detected lines.\n""")\n    code("""from google.colab import files\nuploaded = files.upload()\narchives = [name for name in uploaded if name.lower().endswith(\'.zip\')]\nif len(archives) != 1:\n    raise ValueError(\'Upload exactly one common source/results ZIP.\')\nuploaded_zip = Path(\'/content/bpl_uploaded.zip\')\nuploaded_zip.write_bytes(uploaded[archives[0]])\nROOT = Path(\'/content/bpl_ocr\')\nindex = import_bundle(uploaded_zip, ROOT)\nprint(\'Loaded seven pages. CSV:\', ROOT / \'results/lines.csv\')\n""", "inputs")\n    md("""## 3. Recognize lines\n\n`MAX_NEW_LINES = None` processes all remaining target lines; use `6` for a quick smoke test.\nCompleted outputs are retained by default, so existing Tesseract/mOCR results need no rerun.\nEmpty/failed/truncated outputs are retried. `FORCE = True` re-recognizes target lines;\nearlier raw responses and their original settings remain in the log.\n\nEvery model receives only the original line image. No other OCR text, name list or corrections\nenter recognition. Printed ditto marks and abbreviations are kept as recognized.\nInterrupt the cell if necessary, then run section 4 to download a checkpoint.\n""")\n    code("""MAX_NEW_LINES = None  # Set 6 for a short trial.\nFORCE = False\nif ENGINE != _SETUP_ENGINE:\n    raise RuntimeError(\'Reset the runtime before changing ENGINE.\')\nsummary = run_ocr(ROOT, ENGINE, device=DEVICE, max_new_lines=MAX_NEW_LINES, force=FORCE)\nprint(summary)\n""", "run")\n    md("""## 4. Download the same output folder\n\n`results/lines.csv` contains only the five model text columns and `sources`.\nBlank cells mean no available output; internal status/raw records distinguish unrun and failed cases.\nIt preserves one row per shared detected line, including failures. It does not join people or parse fields.\n\nThe ZIP also retains original scans, shared coordinates and raw outputs for resuming.\nUpload this latest ZIP in a fresh runtime for the next model. Download before Colab discards the VM.\n""")\n    code("""csv_path = export_lines(ROOT)\narchive = pack_bundle(ROOT, Path(\'/content/bpl_ocr.zip\'))\nprint(\'Line output:\', csv_path)\nfiles.download(str(archive))\n""", "export")\n    notebook = nbf.v4.new_notebook(cells=cells, metadata={\n        "kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"},\n        "language_info": {"name": "python"}, "accelerator": "GPU",\n        "colab": {"name": "ocr_colab.ipynb", "gpuType": "T4", "provenance": []}})\n    nbf.validate(notebook)\n    target = Path(__file__).with_name("ocr_colab.ipynb")\n    nbf.write(notebook, target)\n    return target\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", action="store_true", help="Regenerate the single Colab notebook")\n    parser.add_argument("--pack", type=Path, help="Common source/results folder to package")\n    parser.add_argument("--output", type=Path, default=Path(tempfile.gettempdir()) / "bpl_ocr.zip")\n    args = parser.parse_args()\n    if args.notebook:\n        print(build_notebook())\n    elif args.pack:\n        print(pack_bundle(args.pack, args.output))\n    else:\n        parser.print_help()\n'
_CODE = Path('/content/bpl_ocr_code')
_CODE.mkdir(exist_ok=True)
(_CODE / 'ocr_pipeline.py').write_text(_PIPELINE_SOURCE, encoding='utf-8')
sys.path.insert(0, str(_CODE))
from ocr_pipeline import import_bundle, run_ocr, export_lines, pack_bundle

## 2. Upload the common source/results ZIP

Upload a ZIP of the common `fa26/data/ocr/` folder, or the latest ZIP downloaded below.
It contains `source/` and `results/`. Seven scans are kept once; line crops exist in memory only.
The existing coordinates are shared by all models, including any detection mistakes.
Page 161 uses its lower directory body; pages 162–167 retain their existing detected lines.

In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one common source/results ZIP.')
uploaded_zip = Path('/content/bpl_uploaded.zip')
uploaded_zip.write_bytes(uploaded[archives[0]])
ROOT = Path('/content/bpl_ocr')
index = import_bundle(uploaded_zip, ROOT)
print('Loaded seven pages. CSV:', ROOT / 'results/lines.csv')

## 3. Recognize lines

`MAX_NEW_LINES = None` processes all remaining target lines; use `6` for a quick smoke test.
Completed outputs are retained by default, so existing Tesseract/mOCR results need no rerun.
Empty/failed/truncated outputs are retried. `FORCE = True` re-recognizes target lines;
earlier raw responses and their original settings remain in the log.

Every model receives only the original line image. No other OCR text, name list or corrections
enter recognition. Printed ditto marks and abbreviations are kept as recognized.
Interrupt the cell if necessary, then run section 4 to download a checkpoint.

In [ ]:
MAX_NEW_LINES = None  # Set 6 for a short trial.
FORCE = False
if ENGINE != _SETUP_ENGINE:
    raise RuntimeError('Reset the runtime before changing ENGINE.')
summary = run_ocr(ROOT, ENGINE, device=DEVICE, max_new_lines=MAX_NEW_LINES, force=FORCE)
print(summary)

## 4. Download the same output folder

`results/lines.csv` contains only the five model text columns and `sources`.
Blank cells mean no available output; internal status/raw records distinguish unrun and failed cases.
It preserves one row per shared detected line, including failures. It does not join people or parse fields.

The ZIP also retains original scans, shared coordinates and raw outputs for resuming.
Upload this latest ZIP in a fresh runtime for the next model. Download before Colab discards the VM.

In [ ]:
csv_path = export_lines(ROOT)
archive = pack_bundle(ROOT, Path('/content/bpl_ocr.zip'))
print('Line output:', csv_path)
files.download(str(archive))